# 03 — Model Evaluation (PatchCore Anomaly Detection)

This notebook evaluates the PatchCore models:
1. Anomaly score distribution (good vs defective)
2. Threshold-based classification metrics
3. ROC curve and AUC
4. Anomaly heatmaps (correct and misclassified examples)
5. Error analysis — which defect types are hardest to detect?

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

sys.path.insert(0, str(Path.cwd().parent))

from src.dataset import MVTecDataset
from src.model import PatchCoreModel, get_device
from src.visualize import compute_anomaly_heatmap, create_overlay

DATA_ROOT = "../data/raw"
CATEGORIES = ["bottle", "cable", "transistor"]

device = get_device()
print(f"Using device: {device}")

## 1. Load Model and Test Data

In [ ]:
# Load all models and test datasets
models = {}
test_datasets = {}
all_results = {}

for category in CATEGORIES:
    model_path = f"../models/best_{category}.pth"
    model = PatchCoreModel.load(model_path, device=device)
    models[category] = model
    
    test_ds = MVTecDataset(DATA_ROOT, category, split="test")
    test_datasets[category] = test_ds
    counts = test_ds.get_class_counts()
    
    print(f"{category.upper()}: threshold={model.threshold:.4f} | "
          f"Test: {counts['good']} good + {counts['defective']} defective")

    # Compute scores for all test images
    scores = []
    for i in range(len(test_ds)):
        img_tensor, _ = test_ds[i]
        score, _ = model.predict(img_tensor.unsqueeze(0))
        scores.append(score)
    
    y_true = test_ds.labels
    y_pred = [1 if s > model.threshold else 0 for s in scores]
    all_results[category] = {"y_true": y_true, "scores": scores, "y_pred": y_pred}

print("\nAll models loaded!")

## 2. Anomaly Scores & Classification

In [ ]:
# Classification reports for all categories
for category in CATEGORIES:
    r = all_results[category]
    print(f"\n{'='*50}")
    print(f"  {category.upper()}")
    print(f"{'='*50}")
    print(classification_report(r["y_true"], r["y_pred"], 
                                target_names=["Good", "Defective"], zero_division=0))
    
    good_scores = [s for s, l in zip(r["scores"], r["y_true"]) if l == 0]
    bad_scores = [s for s, l in zip(r["scores"], r["y_true"]) if l == 1]
    print(f"  Good images  — Mean: {np.mean(good_scores):.4f} | Max: {np.max(good_scores):.4f}")
    print(f"  Defective    — Mean: {np.mean(bad_scores):.4f} | Min: {np.min(bad_scores):.4f}")
    print(f"  Threshold:     {models[category].threshold:.4f}")

## 3. Score Distribution & Confusion Matrix

In [ ]:
# Score distribution + confusion matrix for each category
fig, axes = plt.subplots(2, len(CATEGORIES), figsize=(6 * len(CATEGORIES), 10))

for i, category in enumerate(CATEGORIES):
    r = all_results[category]
    scores_arr = np.array(r["scores"])
    labels_arr = np.array(r["y_true"])
    threshold = models[category].threshold
    
    # Score distribution
    axes[0, i].hist(scores_arr[labels_arr == 0], bins=25, alpha=0.6, label="Good", color="green")
    axes[0, i].hist(scores_arr[labels_arr == 1], bins=25, alpha=0.6, label="Defective", color="red")
    axes[0, i].axvline(threshold, color="black", linestyle="--", lw=2, label=f"Threshold ({threshold:.2f})")
    axes[0, i].set_title(f"{category.capitalize()} — Score Distribution")
    axes[0, i].set_xlabel("Anomaly Score")
    axes[0, i].set_ylabel("Count")
    axes[0, i].legend()
    
    # Confusion matrix
    cm = confusion_matrix(r["y_true"], r["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Good", "Defective"],
                yticklabels=["Good", "Defective"], ax=axes[1, i])
    axes[1, i].set_xlabel("Predicted")
    axes[1, i].set_ylabel("True")
    axes[1, i].set_title(f"{category.capitalize()} — Confusion Matrix")

plt.tight_layout()
plt.show()

## 4. ROC Curve

In [ ]:
# ROC curves for all categories
fig, ax = plt.subplots(figsize=(7, 7))

for category in CATEGORIES:
    r = all_results[category]
    fpr, tpr, _ = roc_curve(r["y_true"], r["scores"])
    auc = roc_auc_score(r["y_true"], r["scores"])
    ax.plot(fpr, tpr, linewidth=2, label=f"{category.capitalize()} (AUC={auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — PatchCore Anomaly Detection")
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Anomaly heatmaps — correct and misclassified examples per category
for category in CATEGORIES:
    r = all_results[category]
    test_ds = test_datasets[category]
    model = models[category]
    
    for title, indices in [
        ("Correctly Detected Defects", [i for i in range(len(r["y_true"])) if r["y_true"][i] == 1 and r["y_pred"][i] == 1]),
        ("Missed Defects (False Negatives)", [i for i in range(len(r["y_true"])) if r["y_true"][i] == 1 and r["y_pred"][i] == 0]),
        ("False Alarms (False Positives)", [i for i in range(len(r["y_true"])) if r["y_true"][i] == 0 and r["y_pred"][i] == 1]),
    ]:
        if not indices:
            print(f"{category.capitalize()} — {title}: None")
            continue
        
        show = indices[:4]
        fig, axes = plt.subplots(2, len(show), figsize=(5 * len(show), 8))
        if len(show) == 1:
            axes = axes.reshape(2, 1)
        
        for j, idx in enumerate(show):
            image = Image.open(test_ds.image_paths[idx]).convert("RGB")
            heatmap, score = compute_anomaly_heatmap(model, image)
            overlay = create_overlay(image, heatmap)
            
            axes[0, j].imshow(image.resize((224, 224)))
            axes[0, j].set_title(f"{test_ds.defect_types[idx]}")
            axes[0, j].axis("off")
            
            axes[1, j].imshow(overlay)
            axes[1, j].set_title(f"Score: {r['scores'][idx]:.4f}")
            axes[1, j].axis("off")
        
        fig.suptitle(f"{category.capitalize()} — {title}", fontsize=14)
        plt.tight_layout()
        plt.show()

## 6. Error Analysis by Defect Type

In [ ]:
# Error analysis by defect type across all categories
all_records = []
for category in CATEGORIES:
    r = all_results[category]
    test_ds = test_datasets[category]
    for i in range(len(r["y_true"])):
        all_records.append({
            "category": category,
            "defect_type": f"{category}/{test_ds.defect_types[i]}",
            "true_label": r["y_true"][i],
            "pred_label": r["y_pred"][i],
            "correct": r["y_true"][i] == r["y_pred"][i],
            "anomaly_score": r["scores"][i],
        })

results_df = pd.DataFrame(all_records)

# Only look at defective images
defective_df = results_df[results_df["true_label"] == 1]
acc_by_type = defective_df.groupby("defect_type").agg(
    count=("correct", "size"),
    detection_rate=("correct", "mean"),
    avg_score=("anomaly_score", "mean"),
).sort_values("detection_rate")

print("Detection rate by defect type (defective images only):")
print(acc_by_type.to_string())

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["green" if r >= 0.8 else "orange" if r >= 0.5 else "red" for r in acc_by_type["detection_rate"]]
acc_by_type["detection_rate"].plot(kind="barh", ax=ax, color=colors)
ax.set_xlabel("Detection Rate")
ax.set_title("Defect Detection Rate by Type — PatchCore")
ax.axvline(x=0.8, color="gray", linestyle="--", alpha=0.5, label="80% target")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Summary

Key takeaways:

- **PatchCore** uses pretrained WideResNet-50 features + nearest-neighbor distance for anomaly detection — no gradient-based training needed
- **Bottle** achieves the best results (AUC ~0.99) — defects are large and visually distinct
- **Cable** and **transistor** are harder — subtle defects like `poke_insulation` or `bent_lead` produce smaller feature distances
- **Anomaly heatmaps** localize defects without pixel-level supervision — critical for operator trust in Industry 5.0
- **Recall > Precision** for manufacturing QC — missing a defective product is costlier than a false alarm
- **Future improvements:** PaDiM, EfficientAD, or multi-scale feature fusion could improve detection of subtle defects